In [ ]:
                                                        # MCP Integration Patterns
# MCP Integration Patterns refer to the different ways systems connect and interact using the Model Context Protocol (MCP)—a standard used to allow AI 
# models (like LLMs) to communicate with external tools, APIs, and data sources.
# Think of MCP like a bridge between an AI model and real-world systems (databases, APIs, apps). Integration patterns define how this bridge is built 
# and used.

In [11]:
# Without using LLM
#  SCENARIO: “College Smart Assistant System”
#  Background Story

# A college builds an AI-powered student assistant.

# Students can ask:

# “What is my attendance?”
# “What are my marks?”

# Instead of manually checking portals,
# AI fetches it instantly.

In [16]:
import gradio as gr
students = {
    "101": {"name": "Rahul", "attendance": 85, "marks": 78},
    "102": {"name": "Priya", "attendance": 92, "marks": 88}
}

def get_attendance(student_id):
    if student_id in students:
        return f"Attendance: {students[student_id]['attendance']}%"
    return "Student not found"

def get_marks(student_id):
    if student_id in students:
        return f"Marks: {students[student_id]['marks']}"
    return "Student not found"

def secure_access(user_id, requested_id):
    return user_id == requested_id

def mcp_agent(message, student_id, history):

    if history is None:
        history = []

    user_id = student_id

    if not secure_access(user_id, student_id):
        bot_reply = "Access Denied"
    else:
        message_lower = message.lower()

        if "attendance" in message_lower:
            bot_reply = get_attendance(student_id)

        elif "marks" in message_lower:
            bot_reply = get_marks(student_id)

        elif "hello" in message_lower or "hi" in message_lower:
            bot_reply = "Hello! Ask me about attendance or marks."

        else:
            bot_reply = "I can help with attendance or marks."
    history.append({"role": "user", "content": message})
    history.append({"role": "assistant", "content": bot_reply})

    return history, history
with gr.Blocks() as demo:

    gr.Markdown("Student MCP Agent")

    student_id = gr.Textbox(label="Enter Student ID (e.g., 101)")

    chatbot = gr.Chatbot() 
    msg = gr.Textbox(label="Ask your question")

    state = gr.State([])

    msg.submit(
        mcp_agent,
        inputs=[msg, student_id, state],
        outputs=[chatbot, state]
    )
demo.launch()

* Running on local URL:  http://127.0.0.1:7872
* To create a public link, set `share=True` in `launch()`.


In [ ]:
# Using LLM
#  SCENARIO: “College Smart Assistant System”
#  Background Story

# A college builds an AI-powered student assistant.

# Students can ask:

# “What is my attendance?”
# “What are my marks?”

# Instead of manually checking portals,
# AI fetches it instantly.

In [25]:
import os
from dotenv import load_dotenv
import gradio as gr
from groq import Groq

load_dotenv()
groq_api_key = os.getenv("GROQ_API_KEY")

client = Groq(api_key=groq_api_key)

students = {
    "101": {"name": "Rahul", "attendance": 85, "marks": 78},
    "102": {"name": "Priya", "attendance": 92, "marks": 88}
}

def get_attendance(student_id):
    if student_id in students:
        return "Attendance " + str(students[student_id]["attendance"])
    return "Student not found"

def get_marks(student_id):
    if student_id in students:
        return "Marks " + str(students[student_id]["marks"])
    return "Student not found"

def decide_tool(query):
    try:
        response = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[{"role": "user", "content": "choose get_attendance or get_marks query " + query}]
        )
        return response.choices[0].message.content.strip().lower()
    except:
        return "fallback"

def mcp_agent(message, student_id, history):

    if history is None or not isinstance(history, list):
        history = []

    if not student_id:
        bot_reply = "enter id"
    else:
        tool = decide_tool(message)

        if "attendance" in tool:
            bot_reply = get_attendance(student_id)
        elif "marks" in tool:
            bot_reply = get_marks(student_id)
        else:
            if "attendance" in message.lower():
                bot_reply = get_attendance(student_id)
            elif "marks" in message.lower():
                bot_reply = get_marks(student_id)
            else:
                bot_reply = "ask attendance or marks"

    history = history + [
        {"role": "user", "content": str(message)},
        {"role": "assistant", "content": str(bot_reply)}
    ]

    return history, history

with gr.Blocks() as demo:
    student_id = gr.Textbox()
    chatbot = gr.Chatbot()
    msg = gr.Textbox()
    state = gr.State([])

    msg.submit(
        mcp_agent,
        inputs=[msg, student_id, state],
        outputs=[chatbot, state]
    )

demo.launch()

* Running on local URL:  http://127.0.0.1:7879
* To create a public link, set `share=True` in `launch()`.


In [ ]:
# Without using LLM
# SCENARIO: “Hospital Smart Assistant System”
# Background Story
# A large hospital deploys an AI-powered patient assistant.
# Patients can ask:
# - “What is my appointment schedule?”
# - “What are my latest test results?”
# Instead of calling reception or logging into multiple portals,
# AI fetches it instantly, providing secure, real-time updates.

In [17]:
import gradio as gr
patients = {
    "P101": {
        "name": "Rahul Sharma",
        "appointment": "Dr. Mehta - 25 March, 10:30 AM",
        "test_results": "Blood Test: Normal | Sugar: Slightly High"
    },
    "P102": {
        "name": "Priya Verma",
        "appointment": "Dr. Singh - 24 March, 2:00 PM",
        "test_results": "X-Ray: Clear | Vitamin D: Low"
    }
}

# Tool Functions
def get_appointment(patient_id):
    if patient_id in patients:
        return f"Appointment: {patients[patient_id]['appointment']}"
    return "Patient not found"

def get_test_results(patient_id):
    if patient_id in patients:
        return f"Test Results: {patients[patient_id]['test_results']}"
    return "Patient not found"

def secure_access(user_id, requested_id):
    return user_id == requested_id

# MCP Agent Logic
def hospital_agent(message, patient_id, history):

    if history is None:
        history = []

    user_id = patient_id  # simulate login

    # Security Check
    if not secure_access(user_id, patient_id):
        bot_reply = "Access Denied"

    else:
        message_lower = message.lower()

        if "appointment" in message_lower:
            bot_reply = get_appointment(patient_id)

        elif "test" in message_lower or "result" in message_lower:
            bot_reply = get_test_results(patient_id)

        elif "hello" in message_lower or "hi" in message_lower:
            bot_reply = "Hello! I can help with appointments and test results."

        else:
            bot_reply = "Ask me about your appointment or test results."

    history.append({"role": "user", "content": message})
    history.append({"role": "assistant", "content": bot_reply})

    return history, history

with gr.Blocks() as demo:

    gr.Markdown("Hospital Smart Assistant")

    patient_id = gr.Textbox(label="Enter Patient ID (e.g., P101)")

    chatbot = gr.Chatbot()
    msg = gr.Textbox(label="Ask your question")

    state = gr.State([])

    msg.submit(
        hospital_agent,
        inputs=[msg, patient_id, state],
        outputs=[chatbot, state]
    )
demo.launch()

* Running on local URL:  http://127.0.0.1:7873
* To create a public link, set `share=True` in `launch()`.


In [ ]:
# Using LLM
# SCENARIO: “Hospital Smart Assistant System”
#  Background Story
# A large hospital deploys an AI-powered patient assistant.
# Patients can ask:
# - “What is my appointment schedule?”
# - “What are my latest test results?”
# Instead of calling reception or logging into multiple portals,
# AI fetches it instantly, providing secure, real-time updates.

In [27]:
import os
from dotenv import load_dotenv
import gradio as gr
from groq import Groq

load_dotenv()
groq_api_key = os.getenv("GROQ_API_KEY")

client = Groq(api_key=groq_api_key)

patients = {
    "P101": {
        "name": "Rahul",
        "appointment": "Dr Mehta 25 March 1030 AM",
        "test_results": "Blood Normal Sugar High"
    },
    "P102": {
        "name": "Priya",
        "appointment": "Dr Singh 26 March 200 PM",
        "test_results": "Xray Clear Vitamin D Low"
    }
}

def get_appointment(patient_id):
    if patient_id in patients:
        return "Appointment " + patients[patient_id]["appointment"]
    return "Patient not found"

def get_test_results(patient_id):
    if patient_id in patients:
        return "Test Results " + patients[patient_id]["test_results"]
    return "Patient not found"

def decide_tool(query):
    try:
        response = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[{"role": "user", "content": "choose get_appointment or get_test_results query " + query}]
        )
        return response.choices[0].message.content.strip().lower()
    except:
        return "fallback"

def hospital_agent(message, patient_id, history):
    if history is None or not isinstance(history, list):
        history = []

    if not patient_id:
        bot_reply = "enter id"
    else:
        tool = decide_tool(message)

        if "appointment" in tool:
            bot_reply = get_appointment(patient_id)
        elif "test" in tool or "result" in tool:
            bot_reply = get_test_results(patient_id)
        else:
            if "appointment" in message.lower():
                bot_reply = get_appointment(patient_id)
            elif "test" in message.lower() or "result" in message.lower():
                bot_reply = get_test_results(patient_id)
            else:
                bot_reply = "ask appointment or test results"

    history = history + [
        {"role": "user", "content": str(message)},
        {"role": "assistant", "content": str(bot_reply)}
    ]

    return history, history

with gr.Blocks() as demo:
    patient_id = gr.Textbox()
    chatbot = gr.Chatbot()
    msg = gr.Textbox()
    state = gr.State([])

    msg.submit(
        hospital_agent,
        inputs=[msg, patient_id, state],
        outputs=[chatbot, state]
    )

demo.launch()

* Running on local URL:  http://127.0.0.1:7881
* To create a public link, set `share=True` in `launch()`.


In [ ]:
# Without Using LLM
# SCENARIO: “Banking Smart Assistant System”
#  Background Story
# A major bank launches an AI-powered customer assistant.
# Customers can ask:
# - “What is my account balance?”
# - “Show me my last 5 transactions.”
# - “When is my loan EMI due?”
#  Instead of logging into apps or waiting on customer service calls,
#  AI fetches the information instantly, securely, and in real time.

#  Core Idea:
# Just like the hospital and college scenarios, the assistant removes manual checking, centralizes financial data, and makes access instant.
#  Impact:
# - Saves customers time.
# - Reduces load on call centers.
# - Provides personalized financial insights on demand.

In [19]:
import gradio as gr
customers = {
    "C101": {
        "name": "Rahul Sharma",
        "balance": 45230,
        "transactions": [
            "₹2000 - Amazon",
            "₹5000 - Rent",
            "₹1200 - Swiggy",
            "₹3000 - Salary Credit",
            "₹800 - Petrol"
        ],
        "emi_due": "5 April 2026"
    },
    "C102": {
        "name": "Priya Verma",
        "balance": 78210,
        "transactions": [
            "₹1500 - Myntra",
            "₹2500 - Electricity Bill",
            "₹10000 - Salary Credit",
            "₹2000 - Uber",
            "₹900 - Grocery"
        ],
        "emi_due": "10 April 2026"
    }
}
def get_balance(customer_id):
    if customer_id in customers:
        return f"Balance: ₹{customers[customer_id]['balance']}"
    return "Customer not found"

def get_transactions(customer_id):
    if customer_id in customers:
        txns = customers[customer_id]['transactions']
        return "Last 5 Transactions:\n" + "\n".join(txns)
    return "Customer not found"

def get_emi(customer_id):
    if customer_id in customers:
        return f"EMI Due Date: {customers[customer_id]['emi_due']}"
    return "Customer not found"

def secure_access(user_id, requested_id):
    return user_id == requested_id

def banking_agent(message, customer_id, history):

    if history is None:
        history = []

    user_id = customer_id  # simulate login

    # Security Check
    if not secure_access(user_id, customer_id):
        bot_reply = "Access Denied"

    else:
        msg = message.lower()

        if "balance" in msg:
            bot_reply = get_balance(customer_id)

        elif "transaction" in msg:
            bot_reply = get_transactions(customer_id)

        elif "emi" in msg or "loan" in msg:
            bot_reply = get_emi(customer_id)

        elif "hello" in msg or "hi" in msg:
            bot_reply = "Hello! I can help with balance, transactions, and EMI."

        else:
            bot_reply = "Ask me about balance, transactions, or EMI."

    history.append({"role": "user", "content": message})
    history.append({"role": "assistant", "content": bot_reply})

    return history, history

with gr.Blocks() as demo:

    gr.Markdown("Banking Smart Assistant")

    customer_id = gr.Textbox(label="Enter Customer ID (e.g., C101)")

    chatbot = gr.Chatbot()
    msg = gr.Textbox(label="Ask your question")

    state = gr.State([])

    msg.submit(
        banking_agent,
        inputs=[msg, customer_id, state],
        outputs=[chatbot, state]
    )
demo.launch()

* Running on local URL:  http://127.0.0.1:7875
* To create a public link, set `share=True` in `launch()`.


In [ ]:
# Using LLM
# SCENARIO: “Banking Smart Assistant System”
#  Background Story
# A major bank launches an AI-powered customer assistant.
# Customers can ask:
# - “What is my account balance?”
# - “Show me my last 5 transactions.”
# - “When is my loan EMI due?”
#  Instead of logging into apps or waiting on customer service calls,
#  AI fetches the information instantly, securely, and in real time.

#  Core Idea:
# Just like the hospital and college scenarios, the assistant removes manual checking, centralizes financial data, and makes access instant.
#  Impact:
# - Saves customers time.
# - Reduces load on call centers.
# - Provides personalized financial insights on demand.

In [28]:
import os
from dotenv import load_dotenv
import gradio as gr
from groq import Groq

load_dotenv()
groq_api_key = os.getenv("GROQ_API_KEY")

client = Groq(api_key=groq_api_key)

customers = {
    "C101": {
        "name": "Rahul",
        "balance": 45230,
        "transactions": [
            "2000 Amazon",
            "5000 Rent",
            "1200 Food",
            "3000 Salary",
            "800 Petrol"
        ],
        "emi": "5 April 2026"
    },
    "C102": {
        "name": "Priya",
        "balance": 78210,
        "transactions": [
            "1500 Shopping",
            "2500 Electricity",
            "10000 Salary",
            "2000 Travel",
            "900 Grocery"
        ],
        "emi": "10 April 2026"
    }
}

def get_balance(customer_id):
    if customer_id in customers:
        return "Balance " + str(customers[customer_id]["balance"])
    return "Customer not found"

def get_transactions(customer_id):
    if customer_id in customers:
        return "Transactions " + " ".join(customers[customer_id]["transactions"])
    return "Customer not found"

def get_emi(customer_id):
    if customer_id in customers:
        return "EMI Due " + customers[customer_id]["emi"]
    return "Customer not found"

def decide_tool(query):
    try:
        response = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[{"role": "user", "content": "choose get_balance or get_transactions or get_emi query " + query}]
        )
        return response.choices[0].message.content.strip().lower()
    except:
        return "fallback"

def banking_agent(message, customer_id, history):
    if history is None or not isinstance(history, list):
        history = []

    if not customer_id:
        bot_reply = "enter id"
    else:
        tool = decide_tool(message)

        if "balance" in tool:
            bot_reply = get_balance(customer_id)
        elif "transaction" in tool:
            bot_reply = get_transactions(customer_id)
        elif "emi" in tool or "loan" in tool:
            bot_reply = get_emi(customer_id)
        else:
            if "balance" in message.lower():
                bot_reply = get_balance(customer_id)
            elif "transaction" in message.lower():
                bot_reply = get_transactions(customer_id)
            elif "emi" in message.lower() or "loan" in message.lower():
                bot_reply = get_emi(customer_id)
            else:
                bot_reply = "ask balance transactions or emi"

    history = history + [
        {"role": "user", "content": str(message)},
        {"role": "assistant", "content": str(bot_reply)}
    ]

    return history, history

with gr.Blocks() as demo:
    customer_id = gr.Textbox()
    chatbot = gr.Chatbot()
    msg = gr.Textbox()
    state = gr.State([])

    msg.submit(
        banking_agent,
        inputs=[msg, customer_id, state],
        outputs=[chatbot, state]
    )

demo.launch()

* Running on local URL:  http://127.0.0.1:7882
* To create a public link, set `share=True` in `launch()`.


In [ ]:
# SCENARIO: “AI Banking Assistant with Role-Based Access”
# Background Story

# A bank builds an AI assistant to help users:

# Check account balance
# View transactions
# Approve loans
# Manage customer accounts

# But not everyone can do everything

# Roles in the Bank
# Role	Description
# Customer	Bank account holder
# Employee	Bank staff
# Manager	Branch manager
# Permissions (RBAC)
# Action	Customer	Employee	Manager
# View own balance
# View others' accounts
# Approve loan
# View all transactions

In [38]:
import os
from dotenv import load_dotenv
from groq import Groq
import gradio as gr

load_dotenv()
groq_api_key = os.getenv("GROQ_API_KEY")

if not groq_api_key:
    raise ValueError("GROQ_API_KEY not found")

client = Groq(api_key=groq_api_key)

accounts = {
    "1001": {"name": "Amit", "balance": 50000},
    "1002": {"name": "Neha", "balance": 75000}
}

def get_balance(account_id):
    if account_id in accounts:
        return "Balance of " + account_id + " " + str(accounts[account_id]["balance"])
    return "Account not found"

def approve_loan(account_id):
    if account_id in accounts:
        return "Loan approved for account " + account_id
    return "Account not found"

def secure_access(role, user_account, requested_account, action):
    if role == "manager":
        return True
    elif role == "employee":
        if action == "approve_loan":
            return False
        return True
    elif role == "customer":
        return user_account == requested_account and action != "approve_loan"
    return False

def decide_action(query):
    try:
        response = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[{"role": "user", "content": "choose get_balance or approve_loan " + query}]
        )
        return response.choices[0].message.content.strip().lower()
    except:
        return "fallback"

def banking_agent(message, role, user_account, requested_account, history):

    if history is None:
        history = []

    if not user_account:
        bot_reply = "enter account id"
    else:
        if not requested_account:
            requested_account = user_account

        action = decide_action(message)

        if not secure_access(role, user_account, requested_account, action):
            bot_reply = "access denied"
        else:
            if "balance" in action:
                bot_reply = get_balance(requested_account)
            elif "loan" in action:
                bot_reply = approve_loan(requested_account)
            else:
                msg = message.lower()
                if "balance" in msg:
                    bot_reply = get_balance(requested_account)
                elif "loan" in msg:
                    bot_reply = approve_loan(requested_account)
                else:
                    bot_reply = "ask balance or loan"

    history = history + [
        {"role": "user", "content": str(message)},
        {"role": "assistant", "content": str(bot_reply)}
    ]

    return history, history


with gr.Blocks() as demo:

    role = gr.Dropdown(["customer", "employee", "manager"])
    user_account = gr.Textbox()
    requested_account = gr.Textbox()

    chatbot = gr.Chatbot(height=400)
    msg = gr.Textbox()

    state = gr.State([])

    msg.submit(
        banking_agent,
        inputs=[msg, role, user_account, requested_account, state],
        outputs=[chatbot, state]
    )

demo.launch()

* Running on local URL:  http://127.0.0.1:7889
* To create a public link, set `share=True` in `launch()`.


In [ ]:
# SCENARIO: “University Smart Assistant with Role-Based Access”
#  Background Story
# A university deploys an AI-powered academic assistant to help students, faculty, and administrators.
#  Users can ask:
# - “What is my attendance record?”
# - “Show me my exam results.”
# - “Update course schedules.”
# - “Approve new course registrations.”
# But not everyone can do everything — access depends on roles.

In [46]:
import os
from dotenv import load_dotenv
from groq import Groq
import gradio as gr

load_dotenv()
groq_api_key = os.getenv("GROQ_API_KEY")

client = Groq(api_key=groq_api_key)

students = {
    "s101": {"name": "Amit", "attendance": "85%", "result": "Pass"},
    "s102": {"name": "Neha", "attendance": "92%", "result": "Distinction"}
}

def get_attendance(student_id):
    if student_id in students:
        return "Attendance of " + student_id + " " + students[student_id]["attendance"]
    return "Student not found"

def get_result(student_id):
    if student_id in students:
        return "Result of " + student_id + " " + students[student_id]["result"]
    return "Student not found"

def update_schedule():
    return "Course schedule updated"

def approve_registration(student_id):
    if student_id in students:
        return "Registration approved for " + student_id
    return "Student not found"

def secure_access(role, user_id, requested_id, action):
    if role == "admin":
        return True

    if role == "faculty":
        if action == "approve_registration":
            return False
        return True

    if role == "student":
        if action in ["get_attendance", "get_result"]:
            return user_id == requested_id
        return False

    return False

def decide_action(query):
    q = query.lower()

    if "attendance" in q:
        return "get_attendance"
    if "result" in q or "marks" in q:
        return "get_result"
    if "schedule" in q:
        return "update_schedule"
    if "approve" in q or "registration" in q:
        return "approve_registration"

    try:
        response = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[{
                "role": "user",
                "content": "choose one get_attendance get_result update_schedule approve_registration " + query
            }]
        )
        return response.choices[0].message.content.strip().lower()
    except:
        return "fallback"

def university_agent(message, role, user_id, requested_id, history):

    if history is None:
        history = []

    user_id = user_id.strip().lower()
    requested_id = requested_id.strip().lower() if requested_id else user_id

    action = decide_action(message)

    if not secure_access(role, user_id, requested_id, action):
        bot_reply = "access denied"
    else:
        if action == "get_attendance":
            bot_reply = get_attendance(requested_id)
        elif action == "get_result":
            bot_reply = get_result(requested_id)
        elif action == "update_schedule":
            bot_reply = update_schedule()
        elif action == "approve_registration":
            bot_reply = approve_registration(requested_id)
        else:
            bot_reply = "ask valid query"

    history = history + [
        {"role": "user", "content": str(message)},
        {"role": "assistant", "content": str(bot_reply)}
    ]

    return history, history


with gr.Blocks() as demo:

    role = gr.Dropdown(["student", "faculty", "admin"])
    user_id = gr.Textbox(label="Your ID")
    requested_id = gr.Textbox(label="Target Student ID optional")

    chatbot = gr.Chatbot(height=400)
    msg = gr.Textbox()

    state = gr.State([])

    msg.submit(
        university_agent,
        inputs=[msg, role, user_id, requested_id, state],
        outputs=[chatbot, state]
    )

demo.launch()

* Running on local URL:  http://127.0.0.1:7897
* To create a public link, set `share=True` in `launch()`.


In [ ]:
# SCENARIO: “Retail Smart Assistant with Role-Based Access”
#  Background Story
# A large retail chain introduces an AI-powered store assistant to streamline operations.
#  Users can ask:
# - “What is my purchase history?”
# - “Check inventory for product X.”
# - “Approve supplier orders.”
# - “Manage employee schedules.”
#  But not everyone can do everything — access depends on roles.

In [47]:
import gradio as gr

customers = {
    "c101": {"name": "Ravi", "history": "Bought Shoes, Shirt"},
    "c102": {"name": "Priya", "history": "Bought Phone, Bag"}
}

inventory = {
    "shoes": "50 units",
    "phone": "30 units",
    "bag": "20 units"
}

def get_purchase_history(cid):
    if cid in customers:
        return "Purchase history of " + customers[cid]["name"] + " " + customers[cid]["history"]
    return "Customer not found"

def check_inventory(product):
    product = product.lower()
    if product in inventory:
        return product + " stock " + inventory[product]
    return "Product not found"

def approve_order():
    return "Supplier order approved"

def manage_schedule():
    return "Employee schedules updated"

def secure_access(role, user_id, requested_id, action):
    if role == "manager":
        return True

    if role == "staff":
        if action in ["approve_order"]:
            return False
        return True

    if role == "customer":
        if action == "get_history":
            return user_id == requested_id
        return False

    return False

def decide_action(message):
    msg = message.lower()

    if "purchase" in msg or "history" in msg:
        return "get_history"
    if "inventory" in msg or "stock" in msg:
        return "check_inventory"
    if "approve" in msg:
        return "approve_order"
    if "schedule" in msg:
        return "manage_schedule"

    return "unknown"

def extract_product(message):
    words = message.lower().split()
    for w in words:
        if w in inventory:
            return w
    return None

def retail_agent(message, role, user_id, requested_id, history):

    if history is None:
        history = []

    user_id = user_id.strip().lower()
    requested_id = requested_id.strip().lower() if requested_id else user_id

    action = decide_action(message)

    if not secure_access(role, user_id, requested_id, action):
        bot_reply = "access denied"
    else:
        if action == "get_history":
            bot_reply = get_purchase_history(requested_id)

        elif action == "check_inventory":
            product = extract_product(message)
            if product:
                bot_reply = check_inventory(product)
            else:
                bot_reply = "enter product name"

        elif action == "approve_order":
            bot_reply = approve_order()

        elif action == "manage_schedule":
            bot_reply = manage_schedule()

        else:
            bot_reply = "ask valid query"

    history = history + [
        {"role": "user", "content": message},
        {"role": "assistant", "content": bot_reply}
    ]

    return history, history


with gr.Blocks() as demo:

    role = gr.Dropdown(["customer", "staff", "manager"])
    user_id = gr.Textbox(label="Your ID")
    requested_id = gr.Textbox(label="Target Customer ID optional")

    chatbot = gr.Chatbot(height=400)
    msg = gr.Textbox()

    state = gr.State([])

    msg.submit(
        retail_agent,
        inputs=[msg, role, user_id, requested_id, state],
        outputs=[chatbot, state]
    )

demo.launch()

* Running on local URL:  http://127.0.0.1:7898
* To create a public link, set `share=True` in `launch()`.


In [ ]:
# SCENARIO: “Corporate Research Assistant System”
# Background Story
# A multinational company deploys an AI-powered business intelligence assistant.
# Employees can ask:
# - “What’s the latest news about Tesla?”
# - “What’s Tesla’s current stock price?”
# - “Give me a company profile instantly.”
# Instead of manually searching news sites, finance portals, and HR databases,
# AI fetches all the data in parallel, analyzes it, and generates a professional report.

#  How it works (mapped to your pipeline):
# - Parallel Data Collection → AI gathers news, stock prices, and company profiles simultaneously.
# - LLM Analysis → AI interprets the combined data, highlighting key insights.
# - Report Generation → AI produces a polished, executive-ready report.

#  Impact:
# - Saves analysts hours of manual research.
# - Provides real-time, consolidated insights for decision-making.
# - Empowers managers with instant reports for board meetings or investor updates.

In [56]:
import os
from dotenv import load_dotenv
import asyncio
import random
import nest_asyncio

load_dotenv(override=True) 
groq_api_key = os.getenv("GROQ_API_KEY")

if not groq_api_key:
    raise ValueError("GROQ_API_KEY not found! Add it to your .env file.")

from groq import Groq
client = Groq(api_key=groq_api_key)

nest_asyncio.apply()

async def web_search(query):
    await asyncio.sleep(1)  # simulate delay
    return f"News about {query}: Market is growing fast."

async def get_stock_data(company):
    await asyncio.sleep(1)
    price = random.randint(100, 500)
    return f"Stock price of {company}: ${price}"

async def fetch_company_profile(company):
    await asyncio.sleep(1)
    return f"{company} has 5000 employees, HQ in USA"

async def parallel_research(company):
    results = await asyncio.gather(
        web_search(company),
        get_stock_data(company),
        fetch_company_profile(company),
        return_exceptions=True
    )

    news, stock, profile = results

    return {
        "news": news if not isinstance(news, Exception) else "News unavailable",
        "stock": stock if not isinstance(stock, Exception) else "Stock unavailable",
        "profile": profile if not isinstance(profile, Exception) else "Profile unavailable"
    }

def analyse_text(text):
    try:
        response = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[{
                "role": "user",
                "content": f"Analyze this data and give key insights:\n{text}"
            }]
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"Analysis skipped due to API error: {str(e)}"

def generate_report(analysis, company):
    try:
        response = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[{
                "role": "user",
                "content": f"Create a professional report for {company}:\n{analysis}"
            }]
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"Report generation skipped due to API error: {str(e)}"

async def full_pipeline(company):
    # Step 1: Parallel Data Collection
    data = await parallel_research(company)

    combined_text = f"""
    News: {data['news']}
    Stock: {data['stock']}
    Profile: {data['profile']}
    """

    # Step 2: Analysis (LLM)
    analysis = analyse_text(combined_text)

    # Step 3: Report Generation (LLM)
    report = generate_report(analysis, company)

    return report

if __name__ == "__main__":
    company_name = "Tesla"

    # If in Jupyter/Colab use `await full_pipeline(company_name)`
    result = asyncio.run(full_pipeline(company_name))

    print("FINAL REPORT:\n")
    print(result)

FINAL REPORT:

**Tesla, Inc. Market Analysis and Investment Opportunity Report**

**Executive Summary:**

This report provides an analysis of Tesla, Inc.'s current market position, stock performance, and potential for growth. Based on the provided data, our key findings indicate a positive market trend, stable stock price, and opportunities for expansion. With a mid-sized to large company structure and a significant presence in the North American market, Tesla is well-positioned for future growth. However, to gain a more comprehensive understanding of the company's performance and prospects, additional data points are necessary.

**Market Analysis:**

The current market trend for Tesla indicates a growing demand for electric vehicles, expanding product lines, and strategic partnerships. This trend is expected to continue, driven by increasing consumer interest in sustainable energy solutions and government initiatives to promote the adoption of electric vehicles. As a result, Tesla is 

In [ ]:
#  SCENARIO: “Healthcare Research Assistant System”
#  Background Story
# A medical research institute deploys an AI-powered clinical intelligence assistant.
#  Researchers and doctors can ask:
# • 	“What’s the latest research on diabetes treatments?”
# • 	“Summarize recent clinical trial results for cancer drugs.”
# • 	“Give me a profile of a pharmaceutical company instantly.”
# Instead of manually searching journals, trial databases, and company reports,
# AI fetches all the data in parallel, analyzes it, and generates a professional research summary.

#  How it works (mapped to your pipeline):
# • 	Parallel Data Collection → AI gathers medical news, trial results, and company profiles simultaneously.
# • 	LLM Analysis → AI interprets the combined data, highlighting key medical insights.
# • 	Report Generation → AI produces a polished, researcher-ready report.

In [58]:
import os
from dotenv import load_dotenv
import asyncio
import random
import nest_asyncio

load_dotenv(override=True)
groq_api_key = os.getenv("GROQ_API_KEY")

if not groq_api_key:
    raise ValueError("GROQ_API_KEY not found! Add it to your .env file.")

from groq import Groq
client = Groq(api_key=groq_api_key)

nest_asyncio.apply()

async def fetch_medical_news(topic):
    await asyncio.sleep(1)  # simulate delay
    return f"Latest news on {topic}: Multiple breakthroughs reported in top journals."

async def fetch_clinical_trials(topic):
    await asyncio.sleep(1)
    trials = [
        f"Trial on {topic} drug A shows 70% efficacy.",
        f"Trial on {topic} drug B shows 65% efficacy."
    ]
    return " | ".join(trials)

async def fetch_pharma_company_profile(company):
    await asyncio.sleep(1)
    return f"{company}: 12000 employees, HQ in Switzerland, specializes in oncology and vaccines."

async def parallel_healthcare_research(topic, company):
    results = await asyncio.gather(
        fetch_medical_news(topic),
        fetch_clinical_trials(topic),
        fetch_pharma_company_profile(company),
        return_exceptions=True
    )

    news, trials, profile = results

    return {
        "news": news if not isinstance(news, Exception) else "News unavailable",
        "trials": trials if not isinstance(trials, Exception) else "Clinical trials unavailable",
        "profile": profile if not isinstance(profile, Exception) else "Company profile unavailable"
    }

def analyse_healthcare_data(text):
    try:
        response = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[{
                "role": "user",
                "content": f"Analyze this healthcare data and highlight key medical insights:\n{text}"
            }]
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"Analysis skipped due to API error: {str(e)}"

def generate_healthcare_report(analysis, topic, company):
    try:
        response = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[{
                "role": "user",
                "content": f"Create a professional healthcare research report for {topic} and {company}:\n{analysis}"
            }]
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"Report generation skipped due to API error: {str(e)}"

async def healthcare_pipeline(topic, company):
    # Step 1: Parallel Data Collection
    data = await parallel_healthcare_research(topic, company)

    combined_text = f"""
    Medical News: {data['news']}
    Clinical Trials: {data['trials']}
    Company Profile: {data['profile']}
    """

    # Step 2: LLM Analysis
    analysis = analyse_healthcare_data(combined_text)

    # Step 3: Report Generation
    report = generate_healthcare_report(analysis, topic, company)

    return report

if __name__ == "__main__":
    medical_topic = "diabetes"
    pharma_company = "Pfizer"

    result = asyncio.run(healthcare_pipeline(medical_topic, pharma_company))
    print("HEALTHCARE RESEARCH REPORT:\n")
    print(result)

HEALTHCARE RESEARCH REPORT:

**Professional Healthcare Research Report: Diabetes and Pfizer**

**Executive Summary**

This report provides an in-depth analysis of the current state of diabetes research and treatment, with a focus on recent advancements and the role of pharmaceutical companies like Pfizer. Our key findings indicate significant progress in understanding and managing diabetes, with multiple breakthroughs and promising clinical trials underway. We also highlight the potential of diabetes drugs A and B, which have shown efficacy rates of 70% and 65%, respectively. Furthermore, our analysis suggests that Pfizer, a leading pharmaceutical company, may be expanding its therapeutic areas of focus to include diabetes research.

**Introduction**

Diabetes is a chronic and debilitating disease that affects millions of people worldwide. The growing prevalence of diabetes has sparked a surge in research efforts aimed at developing effective treatments and improving patient outcomes. 

In [ ]:
# SCENARIO: “AI IT Helpdesk Assistant in a Large Company”
# Background Story

# A large company (like Infosys or TCS) has thousands of employees.

# Employees face issues daily:

# VPN not working
# System hacked
# Email access denied
# Network outage

# Instead of manual IT support, the company builds an:

# AI IT Helpdesk Assistant

In [59]:
import os
from dotenv import load_dotenv
from groq import Groq
import asyncio
import random
import logging

load_dotenv()

logging.basicConfig(level=logging.INFO)

groq_api_key = os.getenv("GROQ_API_KEY")
client = Groq(api_key=groq_api_key)

# STEP 3: Simulated MCP Tools (Mock APIs)
async def page_security_team(payload):
    return "Security team paged"

async def create_jira_ticket(payload):
    return f"Jira ticket created: {payload}"

async def check_network_status(payload):
    return random.choice(["Network degraded", "All systems normal"])

async def alert_noc_team(payload):
    return "NOC team alerted"

async def get_ad_user(payload):
    return "user_123"

async def reset_permissions(payload):
    return "Permissions reset successfully"

async def query_postgres(payload):
    if random.random() < 0.7:
        raise Exception("Database timeout")
    return "Data from Postgres"

async def query_sqlite_cache(payload):
    return "Data from SQLite cache (fallback)"

# STEP: LLM Classifier via Groq
def classify_issue(issue):
    prompt = f"""
    Classify the following IT issue into ONE category only:
    network, hardware, software, security, access

    Issue: {issue}

    Return only the category name.
    """

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}]
    )

    return response.choices[0].message.content.strip().lower()

# STEP: Conditional Routing Logic
async def smart_it_triage(issue, severity):
    issue_type = classify_issue(issue)

    print(f"Classified as: {issue_type}")

    if issue_type == "security":
        await page_security_team({"issue": issue})
        return await create_jira_ticket({"project": "SEC", "priority": "Blocker"})

    elif issue_type == "network" and severity == "high":
        status = await check_network_status({})
        if "degraded" in status.lower():
            return await alert_noc_team({"issue": issue})
        else:
            return await create_jira_ticket({"project": "NET", "priority": "High"})

    elif issue_type == "access":
        user = await get_ad_user({"query": issue})
        return await reset_permissions({"user": user})

    else:
        return await create_jira_ticket({"project": "IT", "priority": "Medium"})

# STEP: Retry with Exponential Backoff
async def call_tool_with_retry(
    primary_tool,
    fallback_tool=None,
    max_retries=3
):
    delay = 1

    for attempt in range(max_retries):
        try:
            return await primary_tool({})
        except Exception as e:
            logging.warning(f"Attempt {attempt+1} failed: {e}")

            if attempt == max_retries - 1:
                if fallback_tool:
                    logging.info("Switching to fallback tool")
                    return await fallback_tool({})
                raise

            await asyncio.sleep(delay)
            delay *= 2
async def run_demo():
    print("=== Smart IT Triage Demo ===")

    result = await smart_it_triage(
        issue="VPN not working for employee",
        severity="high"
    )

    print("Routing Result:", result)

    print("\n=== Retry Pattern Demo ===")

    data = await call_tool_with_retry(
        query_postgres,
        fallback_tool=query_sqlite_cache
    )

    print("Data Result:", data)

await run_demo()

=== Smart IT Triage Demo ===


INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


Classified as: security
Routing Result: Jira ticket created: {'project': 'SEC', 'priority': 'Blocker'}

=== Retry Pattern Demo ===
Data Result: Data from Postgres


In [ ]:
# SCENARIO: “AI Healthcare Support Assistant in a Large Hospital”
#  Background Story
# A large hospital (like Apollo or Fortis) has thousands of patients and staff members.
#  Daily Challenges Faced:
# - Patients waiting hours for appointment confirmations
# - Confusion about lab test results and reports
# - Doctors overwhelmed with scheduling and follow-up reminders
# - Nurses struggling to track medicine administration times
# - Emergency cases needing instant triage
# Instead of manual coordination, the hospital builds an:
# AI Healthcare Support Assistant
#  Capabilities:
# -  Smart Scheduling: Automatically books and reschedules patient appointments based on doctor availability.
# -  Lab Report Explainer: Summarizes test results in simple language for patients.
# -  Medication Tracker: Sends reminders to nurses and patients about dosage timings.
# -  Emergency Triage: Instantly categorizes incoming cases (critical, urgent, routine) and alerts the right medical team.
# -  Follow-up Automation: Sends personalized recovery instructions and reminders after discharge.
#  Impact:
# - Reduced patient waiting time
# - Doctors spend more time on treatment, less on admin work
# - Nurses avoid errors in medicine administration
# - Faster response in emergencies
# - Improved patient satisfaction and trust

In [60]:
import os
from dotenv import load_dotenv
from groq import Groq
import asyncio
import random
import logging

load_dotenv()
logging.basicConfig(level=logging.INFO)

groq_api_key = os.getenv("GROQ_API_KEY")
if not groq_api_key:
    raise ValueError("GROQ_API_KEY missing in .env")

client = Groq(api_key=groq_api_key)

async def smart_schedule(payload):
    await asyncio.sleep(1)
    return f"Appointment scheduled for patient {payload['patient']} with Dr. {payload['doctor']}"

async def explain_lab_report(payload):
    await asyncio.sleep(1)
    return f"Lab Report Summary for {payload['patient']}: {payload['report']}"

async def medication_reminder(payload):
    await asyncio.sleep(1)
    return f"Reminder sent to {payload['nurse']} for {payload['patient']}'s {payload['medication']}"

async def emergency_triage(payload):
    await asyncio.sleep(1)
    levels = ["critical", "urgent", "routine"]
    triage_level = random.choice(levels)
    return f"Case categorized as {triage_level}. Alert sent to {payload['team']}"

async def followup_instructions(payload):
    await asyncio.sleep(1)
    return f"Follow-up instructions sent to patient {payload['patient']}"

def classify_case(description):
    prompt = f"""
    Classify the following hospital case into ONE category only:
    scheduling, lab, medication, emergency, followup

    Case: {description}

    Return only the category name.
    """
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content.strip().lower()

async def smart_healthcare_triage(case_description, patient):
    case_type = classify_case(case_description)
    print(f"Classified case as: {case_type}")
    if case_type == "scheduling":
        return await smart_schedule({"patient": patient, "doctor": "Dr. Smith"})
    elif case_type == "lab":
        return await explain_lab_report({"patient": patient, "report": "Blood test normal"})
    elif case_type == "medication":
        return await medication_reminder({"nurse": "Nurse Joy", "patient": patient, "medication": "Insulin"})
    elif case_type == "emergency":
        return await emergency_triage({"team": "ER", "patient": patient})
    elif case_type == "followup":
        return await followup_instructions({"patient": patient})
    else:
        return f"No automated workflow available for {case_type}"

async def call_tool_with_retry(primary_tool, fallback_tool=None, max_retries=3):
    delay = 1
    for attempt in range(max_retries):
        try:
            return await primary_tool({})
        except Exception as e:
            logging.warning(f"Attempt {attempt+1} failed: {e}")
            if attempt == max_retries - 1 and fallback_tool:
                logging.info("Switching to fallback tool")
                return await fallback_tool({})
            await asyncio.sleep(delay)
            delay *= 2

async def run_demo():
    patient_name = "John Doe"
    result = await smart_healthcare_triage(
        case_description="Patient needs appointment rescheduling",
        patient=patient_name
    )
    print("Triage Result:", result)
    reminder = await call_tool_with_retry(
        primary_tool=lambda _: medication_reminder({"nurse": "Nurse Joy", "patient": patient_name, "medication": "Insulin"}),
        fallback_tool=lambda _: asyncio.sleep(0.5) or "Fallback reminder sent"
    )
    print("Reminder Result:", reminder)

await run_demo()

INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


Classified case as: scheduling
Triage Result: Appointment scheduled for patient John Doe with Dr. Dr. Smith
Reminder Result: Reminder sent to Nurse Joy for John Doe's Insulin
